# 04 — Ontology-based longitudinal progression analysis (47-subject subset)

This notebook performs an initial ontology-based longitudinal progression analysis on a curated PPMI subset with complete multimodal observations.

**Input**:
- `data/ppmi_47_subjects.tsv`
- `mapping/ppmi_pdon_pmdo_mapping.csv`

**Composite progression target**:
- annualised worsening based on `updrs3_score`, `hy`, and `moca`

**Fixed two-visit design**:
- `V04`, `V06`

**Main outputs**:
- progression cohort summary
- subject-level progression table
- progression groups (`slow` / `fast`)
- domain-level descriptive summaries
- figures saved to `output/trajectory_analysis/`


In [ ]:
!pip -q install pandas numpy matplotlib rdflib

## 1) Project root (Drive recommended)

In [ ]:
from pathlib import Path
import os

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/ppmi-ontology-alignment')
else:
    PROJECT_DIR = Path('/content/ppmi-ontology-alignment')

DATA_DIR = PROJECT_DIR / 'data'
MAP_DIR = PROJECT_DIR / 'mapping'
OUT_DIR = PROJECT_DIR / 'output'
TRAJ_OUT_DIR = OUT_DIR / 'trajectory_analysis'
FIG_DIR = TRAJ_OUT_DIR / 'figures'
for p in [DATA_DIR, MAP_DIR, OUT_DIR, TRAJ_OUT_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / 'ppmi_47_subjects.tsv'
MAP_PATH  = MAP_DIR / 'ppmi_pdon_pmdo_mapping.csv'

print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_PATH:', DATA_PATH, 'exists=', DATA_PATH.exists())
print('MAP_PATH :', MAP_PATH, 'exists=', MAP_PATH.exists())
print('TRAJ_OUT_DIR:', TRAJ_OUT_DIR)

## 2) Load mapping and analysis subset

In [ ]:
import pandas as pd
import numpy as np

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing input file: {DATA_PATH}')
if not MAP_PATH.exists():
    raise FileNotFoundError(f'Missing mapping file: {MAP_PATH}')

mapping = pd.read_csv(MAP_PATH, dtype=str).fillna('')
mapping['Variable'] = mapping['Variable'].astype(str).str.strip()
mapping['MappingBucket'] = mapping['MappingBucket'].astype(str).str.strip()
mapping['Category'] = mapping['Category'].astype(str).str.strip()
mapping['TargetIRI'] = mapping['TargetIRI'].astype(str).str.strip()
mapping['Confidence'] = mapping['Confidence'].astype(str).str.strip()

raw = pd.read_csv(DATA_PATH, sep='\t', dtype=str)
print('Rows:', len(raw), 'Cols:', len(raw.columns))
print('Unique PATNO:', raw['PATNO'].astype(str).nunique() if 'PATNO' in raw.columns else 'PATNO missing')
raw.head(3)

## 3) Define core variables and semantic domains

In [ ]:
SUBJECT_META = ['PATNO', 'COHORT', 'subgroup', 'SEX', 'EDUCYRS', 'fampd_bin', 'APOE']
VISIT_META = ['EVENT_ID', 'YEAR', 'visit_date', 'age_at_visit']
REQUIRED_VISITS = ['V04', 'V06']

# Variables used to define the progression label
LABEL_VARS = ['updrs3_score', 'hy', 'moca']

MOTOR_VARS = ['updrs1_score', 'updrs2_score', 'updrs3_score', 'updrs4_score', 'hy']
COGNITION_VARS = ['moca', 'cogstate', 'MCI_testscores']
BIOMARKER_VARS = ['abeta', 'ptau', 'asyn']
IMAGING_VARS = [
    'MIA_LOWPUT_EXPECTED',
    'MIA_CAUDATE_L', 'MIA_CAUDATE_R', 'MIA_CAUDATE_BILAT',
    'MIA_PUTAMEN_L', 'MIA_PUTAMEN_R', 'MIA_PUTAMEN_BILAT',
    'MIA_STRIATUM_L', 'MIA_STRIATUM_R', 'MIA_STRIATUM_BILAT'
]
DIAGNOSIS_VARS = ['PRIMDIAG']

ALL_ANALYSIS_VARS = SUBJECT_META + VISIT_META + DIAGNOSIS_VARS + MOTOR_VARS + COGNITION_VARS + BIOMARKER_VARS + IMAGING_VARS
AVAILABLE_ANALYSIS_VARS = [c for c in ALL_ANALYSIS_VARS if c in raw.columns]
missing_for_analysis = [c for c in ALL_ANALYSIS_VARS if c not in raw.columns]

print('Available analysis vars:', len(AVAILABLE_ANALYSIS_VARS))
print('Missing analysis vars:', missing_for_analysis)
print('Required visits:', REQUIRED_VISITS)
print('Label-defining variables:', LABEL_VARS)

## 4) Basic cohort validation

We require:
- subject identifier (`PATNO`)
- visit identifier (`EVENT_ID`)
- longitudinal index (`YEAR`)
- the three label-defining variables (`updrs3_score`, `hy`, `moca`)

This notebook is designed for a fixed two-visit subset with complete multimodal data.

In [ ]:
required_cols = ['PATNO', 'EVENT_ID', 'YEAR'] + LABEL_VARS
missing_required = [c for c in required_cols if c not in raw.columns]
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

cohort = raw[AVAILABLE_ANALYSIS_VARS].copy()

for c in ['PATNO', 'EVENT_ID', 'YEAR'] + LABEL_VARS:
    cohort[c] = cohort[c].astype(str).str.strip()

cohort = cohort[(cohort['PATNO'] != '') & (cohort['EVENT_ID'] != '')].copy()

visit_counts = cohort.groupby('PATNO')['EVENT_ID'].nunique().sort_values()
visit_counts_df = visit_counts.reset_index(name='n_visits')
print('Subjects:', cohort['PATNO'].nunique())
print('Visit count distribution:')
print(visit_counts.value_counts().sort_index())
visit_counts_df.head()

## 5) Build the longitudinal progression cohort

We retain subjects with the fixed visit pair `V04` and `V06`, together with non-missing values for the variables used to define the progression label.

In [ ]:
def to_float_safe(x):
    s = str(x).strip()
    if s == '' or s.lower() == 'nan':
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def normalize_event_id(x):
    return str(x).strip().upper()


cohort['YEAR_num'] = cohort['YEAR'].apply(to_float_safe)
cohort['EVENT_ID_norm'] = cohort['EVENT_ID'].apply(normalize_event_id)
cohort['updrs3_num'] = cohort['updrs3_score'].apply(to_float_safe)
cohort['hy_num'] = cohort['hy'].apply(to_float_safe)
cohort['moca_num'] = cohort['moca'].apply(to_float_safe)

traj = cohort[
    ~cohort['YEAR_num'].isna() &
    ~cohort['updrs3_num'].isna() &
    ~cohort['hy_num'].isna() &
    ~cohort['moca_num'].isna()
].copy()
traj = traj[traj['EVENT_ID_norm'].isin(REQUIRED_VISITS)].copy()

visit_sets = traj.groupby('PATNO')['EVENT_ID_norm'].apply(lambda x: set(x)).reset_index(name='visit_set')
eligible_subjects = visit_sets[visit_sets['visit_set'].apply(lambda s: set(REQUIRED_VISITS).issubset(s))]['PATNO'].tolist()
traj2 = traj[traj['PATNO'].isin(eligible_subjects)].copy()

traj2 = traj2.sort_values(['PATNO', 'YEAR_num', 'EVENT_ID_norm']).copy()
traj2 = traj2.drop_duplicates(subset=['PATNO', 'EVENT_ID_norm'], keep='first').copy()

visit_counts_fixed = traj2.groupby('PATNO')['EVENT_ID_norm'].nunique()
valid_subjects = visit_counts_fixed[visit_counts_fixed == len(REQUIRED_VISITS)].index.tolist()
traj2 = traj2[traj2['PATNO'].isin(valid_subjects)].copy()

visit_order_map = {'V04': 1, 'V06': 2}
traj2['visit_order'] = traj2['EVENT_ID_norm'].map(visit_order_map)
traj2 = traj2.sort_values(['PATNO', 'visit_order']).copy()

print('Progression cohort subjects:', traj2['PATNO'].nunique())
print('Progression cohort rows:', len(traj2))
print('Expected rows = subjects x 2:', traj2['PATNO'].nunique() * 2)
traj2[['PATNO', 'EVENT_ID', 'EVENT_ID_norm', 'YEAR', 'updrs3_score', 'hy', 'moca']].head(12)

## 6) Derive subject-level progression summaries

For each subject, the notebook computes annualised change in:
- `updrs3_score` (higher = worse)
- `hy` (higher = worse)
- `moca` (lower = worse)

The three annualised changes are then standardised and combined into a composite progression score.

In [ ]:
rows = []
for patno, g in traj2.groupby('PATNO'):
    g = g.sort_values('visit_order')
    first = g[g['EVENT_ID_norm'] == 'V04'].iloc[0]
    last = g[g['EVENT_ID_norm'] == 'V06'].iloc[0]

    year_diff = float(last['YEAR_num']) - float(first['YEAR_num'])
    if year_diff == 0:
        continue

    updrs3_change = float(last['updrs3_num']) - float(first['updrs3_num'])
    hy_change = float(last['hy_num']) - float(first['hy_num'])
    moca_change = float(last['moca_num']) - float(first['moca_num'])

    annualised_updrs3_change = updrs3_change / year_diff
    annualised_hy_change = hy_change / year_diff
    annualised_moca_change = moca_change / year_diff

    rows.append({
        'PATNO': patno,
        'first_event': first['EVENT_ID_norm'],
        'last_event': last['EVENT_ID_norm'],
        'first_year': float(first['YEAR_num']),
        'last_year': float(last['YEAR_num']),
        'year_diff': year_diff,
        'first_updrs3': float(first['updrs3_num']),
        'last_updrs3': float(last['updrs3_num']),
        'first_hy': float(first['hy_num']),
        'last_hy': float(last['hy_num']),
        'first_moca': float(first['moca_num']),
        'last_moca': float(last['moca_num']),
        'updrs3_change': updrs3_change,
        'hy_change': hy_change,
        'moca_change': moca_change,
        'annualised_updrs3_change': annualised_updrs3_change,
        'annualised_hy_change': annualised_hy_change,
        'annualised_moca_change': annualised_moca_change,
        'annualised_moca_worsening': -annualised_moca_change,
        'n_visits_used': len(g),
        'first_primdiag': first['PRIMDIAG'] if 'PRIMDIAG' in g.columns else ''
    })

subject_progression = pd.DataFrame(rows)
print('Subjects with valid progression summaries:', len(subject_progression))
subject_progression.head()

## 7) Build the composite progression score

The three annualised worsening components are standardised using z-scores and then averaged to obtain a composite progression score.

Higher composite scores correspond to faster overall progression.

In [ ]:
def zscore_series(s):
    s = pd.to_numeric(s, errors='coerce')
    mean_ = s.mean()
    std_ = s.std(ddof=0)
    if std_ == 0 or pd.isna(std_):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mean_) / std_

subject_progression['z_updrs3_worsening'] = zscore_series(subject_progression['annualised_updrs3_change'])
subject_progression['z_hy_worsening'] = zscore_series(subject_progression['annualised_hy_change'])
subject_progression['z_moca_worsening'] = zscore_series(subject_progression['annualised_moca_worsening'])

subject_progression['composite_progression_score'] = subject_progression[
    ['z_updrs3_worsening', 'z_hy_worsening', 'z_moca_worsening']
].mean(axis=1)

subject_progression = subject_progression.sort_values('composite_progression_score').reset_index(drop=True)
subject_progression.head(10)

## 8) Assign progression groups

Subjects are grouped into two classes based on the composite progression score:
- `slow`
- `fast`

The split is balanced by construction.

In [ ]:
labels = ['slow', 'fast']
subject_progression['progression_group'] = pd.qcut(
    subject_progression['composite_progression_score'],
    q=2,
    labels=labels,
    duplicates='drop'
)

if subject_progression['progression_group'].isna().any():
    ranked = subject_progression['composite_progression_score'].rank(method='first')
    subject_progression['progression_group'] = pd.qcut(ranked, q=2, labels=labels)

print(subject_progression['progression_group'].value_counts(dropna=False))
subject_progression[['PATNO', 'annualised_updrs3_change', 'annualised_hy_change', 'annualised_moca_worsening', 'composite_progression_score', 'progression_group']].head(10)

## 9) Merge progression groups back to visit-level data

In [ ]:
traj2 = traj2.merge(
    subject_progression[['PATNO', 'progression_group', 'composite_progression_score']],
    on='PATNO',
    how='left'
)
traj2[['PATNO', 'EVENT_ID_norm', 'YEAR', 'updrs3_score', 'hy', 'moca', 'progression_group']].head(12)

## 10) Cohort and progression summaries

In [ ]:
cohort_summary = pd.DataFrame([
    {'metric': 'n_subjects', 'value': int(traj2['PATNO'].nunique())},
    {'metric': 'n_rows', 'value': int(len(traj2))},
    {'metric': 'n_visits_per_subject_expected', 'value': 2},
    {'metric': 'mean_first_updrs3', 'value': round(subject_progression['first_updrs3'].mean(), 3)},
    {'metric': 'mean_last_updrs3', 'value': round(subject_progression['last_updrs3'].mean(), 3)},
    {'metric': 'mean_first_hy', 'value': round(subject_progression['first_hy'].mean(), 3)},
    {'metric': 'mean_last_hy', 'value': round(subject_progression['last_hy'].mean(), 3)},
    {'metric': 'mean_first_moca', 'value': round(subject_progression['first_moca'].mean(), 3)},
    {'metric': 'mean_last_moca', 'value': round(subject_progression['last_moca'].mean(), 3)},
    {'metric': 'mean_composite_progression_score', 'value': round(subject_progression['composite_progression_score'].mean(), 3)}
])

progression_summary = subject_progression.groupby('progression_group').agg(
    n_subjects=('PATNO', 'nunique'),
    mean_first_updrs3=('first_updrs3', 'mean'),
    mean_last_updrs3=('last_updrs3', 'mean'),
    mean_updrs3_change=('annualised_updrs3_change', 'mean'),
    mean_first_hy=('first_hy', 'mean'),
    mean_last_hy=('last_hy', 'mean'),
    mean_hy_change=('annualised_hy_change', 'mean'),
    mean_first_moca=('first_moca', 'mean'),
    mean_last_moca=('last_moca', 'mean'),
    mean_moca_worsening=('annualised_moca_worsening', 'mean'),
    mean_composite_score=('composite_progression_score', 'mean')
).reset_index()

print('--- Cohort summary ---')
print(cohort_summary)
print('\n--- Progression summary ---')
print(progression_summary)

## 11) Ontology-grounded descriptive summaries by domain

Domain summaries are computed at the first visit of the fixed pair (`V04`).

In [ ]:
def safe_mean(df, cols):
    out = {}
    for c in cols:
        if c in df.columns:
            vals = df[c].apply(to_float_safe)
            out[c] = vals.mean(skipna=True)
    return out

first_rows = traj2[traj2['EVENT_ID_norm'] == 'V04'].copy()

first_domain_rows = []
for grp, g in first_rows.groupby('progression_group'):
    row = {'progression_group': grp, 'n_subjects': g['PATNO'].nunique()}
    row.update({f'motor__{k}': v for k, v in safe_mean(g, [c for c in MOTOR_VARS if c in g.columns]).items()})
    row.update({f'cognition__{k}': v for k, v in safe_mean(g, [c for c in COGNITION_VARS if c in g.columns]).items()})
    row.update({f'biomarker__{k}': v for k, v in safe_mean(g, [c for c in BIOMARKER_VARS if c in g.columns]).items()})
    row.update({f'imaging__{k}': v for k, v in safe_mean(g, [c for c in IMAGING_VARS if c in g.columns]).items()})
    first_domain_rows.append(row)

first_domain_summary = pd.DataFrame(first_domain_rows)
first_domain_summary

## 12) Diagnosis profile by progression group

In [ ]:
if 'PRIMDIAG' in first_rows.columns:
    diagnosis_profile = (
        first_rows.groupby(['progression_group', 'PRIMDIAG'])['PATNO']
        .nunique()
        .reset_index(name='n_subjects')
        .sort_values(['progression_group', 'n_subjects'], ascending=[True, False])
    )
else:
    diagnosis_profile = pd.DataFrame(columns=['progression_group', 'PRIMDIAG', 'n_subjects'])

diagnosis_profile.head(20)

## 13) Progression visualisations

In [ ]:
import matplotlib.pyplot as plt

# Figure 1: subject-level UPDRS-III change lines by progression group
plt.figure(figsize=(8, 5))
for grp in ['slow', 'fast']:
    ggrp = traj2[traj2['progression_group'] == grp]
    for patno, g in ggrp.groupby('PATNO'):
        g = g.sort_values('visit_order')
        plt.plot(g['visit_order'], g['updrs3_num'], alpha=0.5)
plt.xticks([1, 2], REQUIRED_VISITS)
plt.xlabel('Visit')
plt.ylabel('updrs3_score')
plt.title('Two-visit longitudinal change in updrs3_score')
plt.tight_layout()
plt.savefig(FIG_DIR / 'updrs3_subject_change_lines.png', dpi=200)
plt.show()

# Figure 2: mean composite-label components by progression group
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for idx, var in enumerate(['annualised_updrs3_change', 'annualised_hy_change', 'annualised_moca_worsening']):
    plot_data = [
        subject_progression.loc[subject_progression['progression_group'] == grp, var].dropna().values
        for grp in ['slow', 'fast']
    ]
    axes[idx].boxplot(plot_data, labels=['slow', 'fast'])
    axes[idx].set_title(var)
plt.tight_layout()
plt.savefig(FIG_DIR / 'composite_progression_components_boxplots.png', dpi=200)
plt.show()

# Figure 3: composite progression score distribution
plot_data = [
    subject_progression.loc[subject_progression['progression_group'] == grp, 'composite_progression_score'].dropna().values
    for grp in ['slow', 'fast']
]
plt.figure(figsize=(7, 5))
plt.boxplot(plot_data, labels=['slow', 'fast'])
plt.ylabel('Composite progression score')
plt.title('Distribution of composite progression score by group')
plt.tight_layout()
plt.savefig(FIG_DIR / 'composite_progression_score_boxplot.png', dpi=200)
plt.show()

print('Saved figures to:', FIG_DIR)

## 14) Export results

In [ ]:
cohort_summary.to_csv(TRAJ_OUT_DIR / 'cohort_summary.csv', index=False)
progression_summary.to_csv(TRAJ_OUT_DIR / 'progression_summary.csv', index=False)
subject_progression.to_csv(TRAJ_OUT_DIR / 'subject_progression_table.csv', index=False)
traj2.to_csv(TRAJ_OUT_DIR / 'progression_visit_level_table.csv', index=False)
first_domain_summary.to_csv(TRAJ_OUT_DIR / 'first_visit_domain_summary.csv', index=False)
diagnosis_profile.to_csv(TRAJ_OUT_DIR / 'diagnosis_profile_by_group.csv', index=False)

print('Wrote:')
for fn in [
    'cohort_summary.csv',
    'progression_summary.csv',
    'subject_progression_table.csv',
    'progression_visit_level_table.csv',
    'first_visit_domain_summary.csv',
    'diagnosis_profile_by_group.csv'
]:
    print('-', TRAJ_OUT_DIR / fn)

## 15) Quick interpretation-ready outputs

In [ ]:
print('--- progression_summary ---')
print(progression_summary.round(3).to_string(index=False))

print('\n--- diagnosis_profile_by_group (top rows) ---')
print(diagnosis_profile.head(20).to_string(index=False))

print('\n--- first_visit_domain_summary ---')
print(first_domain_summary.round(3).to_string(index=False))

print('\n--- composite score components (head) ---')
print(subject_progression[['PATNO', 'annualised_updrs3_change', 'annualised_hy_change', 'annualised_moca_worsening', 'composite_progression_score', 'progression_group']].head(10).round(3).to_string(index=False))